In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, rand, floor, concat, length, trim, to_date
from pyspark.sql.functions import current_date, col
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, lower, trim

In [0]:
# this is only for basic cleaning
class BaseCleaning:
    try:
        def __init__(self,df):
            self.df = df
        def lower_trim(self):
            for fields in self.df.schema:
                if isinstance(fields.dataType, StringType):
                    self.df = self.df.withColumn(fields.name,lower(trim(col(fields.name))))
            print("lower and trim is done")
    except Exception as e:
        print("you have erroe in base cleaning",e)

In [0]:
class Coustomer_info(BaseCleaning):
    try:
        def Coustmer_changes(self):
            renames = {
                            "cst_id":"coustmer_id",
                            "cst_key":"coustmer_key",
                            "cst_firstname":"coustmer_firstname",
                            "cst_lastname":"coustmer_lastname",
                            "cst_marital_status":"marriage_status",
                            "cst_gndr":"coustmer_gender",
                            "cst_create_date":"coustmer_join_date"
                }
            for old,new in renames.items():
                self.df = self.df.withColumnRenamed(old,new)
            #filtering Null values
            self.df = self.df.filter(col("coustmer_id").isNotNull())
            self.df = self.df.dropDuplicates(["coustmer_id"])
            #normilaation
            self.df = self.df.withColumn(
                                            "marriage_status",
                                                when(trim(col("marriage_status")) == "s","single")
                                                .when(trim(col("marriage_status")) == "m","married")
                                                .otherwise("n/a")
            )
            self.df = self.df.withColumn( "coustmer_gender",
                                                when(trim(col("coustmer_gender")) == "m","male")
                                                .when(trim(col("coustmer_gender")) == "F","female")
                                                .otherwise("n/a")

            )
            self.df = self.df.withColumn("data_added",current_date())
            print("sending the data to silver.cleaned_coustmer")
            self.df =  self.df.write.mode("append").format("delta").saveAsTable("lakehouse.sliver.cleaned_coustmer")
            return self.df
    except Exception as e:
        print("you have error in coustmer_change",e)

In [0]:
 class Product_info(BaseCleaning):
    def Product_Cleaning(self):
        try:
            RENAME_MAP = {
            "prd_id": "product_id",
            "cat_id": "category_id",
            "prd_key": "product_number",
            "prd_nm": "product_name",
            "prd_cost": "product_cost",
            "prd_line": "product_line",
            "prd_start_dt": "start_date",
            "prd_end_dt": "end_date",
            "sls_prd_key":"sales_product_key"

            }
            #extracting data to match with sales data and px_cat,
            self.df = self.df.withColumn("cat_id",regexp_replace(substring(col("prd_key"),1,5),"-","_"))
            self.df = self.df.withColumn("sls_prd_key",substring(col("prd_key"),7,length(col("prd_key"))))
            self.df = self.dfwithColumn("prd_cost",coalesce(col("prd_cost"),lit(0)))
            self.df = self.df.withColumn("prd_line",
                                    when(upper(col("prd_line")) == "M","Mountain")
                                    .when(trim(upper(col("prd_line"))) == "R","Road")
                                    .when(upper(col("prd_line")) == "S","Other Sales")
                                    .when(upper(col("prd_line")) == "T","Touring")
                                    .otherwise("n/a")
            )
            windows = Window.partitionBy("prd_start_dt").orderBy("prd_start_dt")
            #adjusting the end date 
            self.df = self.df.withColumn("prd_end_dt",lead("prd_start_dt").over(windows)-1)
            # changing into date
            for fields in self.df.schema:
                if isinstance(fields.dataType,DateType()):
                    self.df = self.df.withColumn(fields,to_date(col("fields"),"yyyy-MM-dd"))
            for old,key in RENAME_MAP.items():
                self.df = self.df.withColumnRenamed(old,new)
            self.df = self.df.withColumn("data_added",current_date())
            print("sending the data into silver layer")
            self.df.write.format("delta").option("mode","append").saveAsTable("lakehouse.silver.crm_product")
        except Exception as e:
            print("you have error at product_info_class",e)
            

In [0]:
#main class
bronze_db = "lakehouse.bronze"
tables = spark.catalog.listTables(bronze_db)
for i in tables:
    df = spark.read.table(f"{bronze_db}.{i.name}")
    if i.name == "cust_info":
        obj = Coustomer_info(df)
        obj.lower_trim()
        new_obj = obj.Coustmer_changes()
        #new_obj.show()

In [0]:
new_df = spark.read.table("lakehouse.sliver.cleaned_coustmer").show()

In [0]:
new_df.select("coustmer_gender").distinct().show()